1. Loading candidate profiles from a parquet file
2. Converting each row into a clean profile string (skipped if `profile_text` already exists)
3. Embedding profile text (resumable via memmap checkpoint)
4. Creating Elasticsearch index and bulk-uploading profiles + embeddings
5. Verifying document count and kNN search

In [1]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: False


In [2]:
import sys
from pathlib import Path

import pyarrow.parquet as pq

sys.path.insert(0, str(Path('..').resolve()))

from core.config import EmbeddingConfig, ElasticsearchConfig, VectorIndexConfig
from core.embeddings import embed_all, embed_texts
from core.elasticsearch import (
    bulk_index,
    candidate_actions,
    create_index,
    get_es_client,
    knn_search,
)
from core.indices import CANDIDATE_INDEX, candidate_mappings
from core.profiles import row_to_candidate_profile

DATASET_PATH = Path('../data/candidate_profiles.parquet')
ID_COLUMN = 'id'
MAX_ROWS = None  # set an int for a smaller test run
EMBEDDINGS_PATH = Path('../data/candidate_embeddings.npy')
PROGRESS_PATH = Path('../data/embed_progress.json')
ES_INDEX = CANDIDATE_INDEX

embed_config = EmbeddingConfig.from_env()
index_config = VectorIndexConfig.from_env(dim=embed_config.dim)
es_config = ElasticsearchConfig.from_env()

In [ ]:
df = pq.read_table(DATASET_PATH).to_pandas()

print(f'Loaded rows: {len(df):,}')
print('Columns:', list(df.columns))

In [ ]:
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

print(f'Kept rows: {len(df):,}')

In [ ]:
if 'profile_text' not in df.columns:
    df['profile_text'] = df.apply(row_to_candidate_profile, axis=1)

profile_texts = df['profile_text'].tolist()
print(f'Profile texts ready: {len(profile_texts):,}')

In [ ]:
candidate_embeddings = embed_all(
    profile_texts,
    EMBEDDINGS_PATH,
    PROGRESS_PATH,
    embed_config,
)

In [ ]:
es = get_es_client(es_config)
info = es.info()
print(f"Connected to Elasticsearch {info['version']['number']}")

create_index(
    es,
    ES_INDEX,
    candidate_mappings(index_config),
    vector_config=index_config,
)

In [ ]:
bulk_index(
    es,
    ES_INDEX,
    candidate_actions(df, candidate_embeddings, index=ES_INDEX, id_column=ID_COLUMN),
)

In [ ]:
doc_count = es.count(index=ES_INDEX)['count']
print(f'Documents in {ES_INDEX}: {doc_count:,} (expected {len(df):,})')

query_text = 'senior python backend developer'
query_vec = embed_texts([query_text], embed_config)[0]

results = knn_search(
    es,
    ES_INDEX,
    query_vec,
    k=5,
    num_candidates=100,
    source=['id', 'position', 'primary_keyword', 'experience_years'],
)

print(f"\nTop matches for: {query_text!r}")
for hit in results['hits']['hits']:
    source = hit['_source']
    print(
        f"  score={hit['_score']:.4f}  "
        f"id={source['id']}  "
        f"position={source.get('position', '')!r}  "
        f"keyword={source.get('primary_keyword', '')!r}"
    )